In [1]:
import anndata as ad
import pandas as pd
import numpy as np
import mofax as mfx
import duckdb as db
def limpiar_drug(s):
    return (s.str.strip()
             .str.lower()
             .str.replace(' ', '_', regex=False)  
             .str.replace(r'_+', '_', regex=True)
             .str.strip('_'))

In [2]:


# --- RUTAS ---
MOFA_MODEL = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/modelo_prueba_final_fast_convergence/modelo_mofa_30factors.hdf5'
INPUT_PARQUET = "/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/datos/datos_con_placa_14/tidy_final.parquet"
DRUG_PARQUET = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/drug.parquet'
OUTPUT_H5AD = '/mnt/lustre/scratch/nlsas/home/ulc/co/mao/modelo_prueba_final_fast_convergence/adata/mofa_adata_30f.h5ad'

# 1. Cargar modelo MOFA
print("Cargando modelo MOFA...")
model = mfx.mofa_model(MOFA_MODEL)

# 2. Factores (muestras x factores)
Z = model.get_factors(df=True)
print(f"Factores: {Z.shape}")
print(f"Índice ejemplo: {Z.index[:3].tolist()}")

# 3. Metadata desde el índice de los factores
# sample formato: drug_concentracion_plate (plate es el último segmento tras _)
split = Z.index.to_series().str.rsplit('_', n=1, expand=True)
plate = split[1]
drug_concentracion = split[0]

# concentración es el último segmento de drug_concentracion
split_conc = drug_concentracion.str.rsplit('_', n=1, expand=True)
concentration = split_conc[1]
drug = split_conc[0]

obs = pd.DataFrame({
    'drug': limpiar_drug(drug).values,
    'concentration': concentration.values,
    'plate': plate.values,
}, index=Z.index)

# 4. Añadir MOA desde drug metadata
print("Añadiendo MOA...")
drug_meta = pd.read_parquet(DRUG_PARQUET)
drug_meta['drug'] = limpiar_drug(drug_meta['drug'])

if 'moa-fine' in drug_meta.columns:
    moa_map = drug_meta[['drug', 'moa-fine']].drop_duplicates('drug')
    obs = obs.merge(moa_map, on='drug', how='left')
    obs.index = Z.index
    print(f"  MOA añadido. NaN: {obs['moa-fine'].isna().sum()}")
else:
    print(f"  Columnas disponibles en drug_meta: {drug_meta.columns.tolist()}")
    print("  Ajusta el nombre de la columna MOA manualmente")

# 5. Pesos de MOFA
print("Extrayendo pesos MOFA...")
W = model.get_weights(df=True)
print(f"Tipo de W: {type(W)}")

# 6. Crear AnnData
print("Creando AnnData...")
adata = ad.AnnData(
    X=Z.values,
    obs=obs,
    var=pd.DataFrame(index=Z.columns),
)
# 7. Guardar pesos en uns
if isinstance(W, dict):
    for view, w in W.items():
        adata.uns[f'mofa_weights_{view}'] = w.values
        adata.uns[f'mofa_weights_genes_{view}'] = w.index.tolist()
    adata.uns['mofa_weights_factors'] = list(Z.columns)
    adata.uns['mofa_views'] = list(W.keys())
else:
    adata.uns['mofa_weights'] = W.values
    adata.uns['mofa_weights_genes'] = W.index.tolist()
    adata.uns['mofa_weights_factors'] = W.columns.tolist()
    adata.uns['mofa_views'] = list(model.get_views())


# 9. Guardar
print(f"\n{adata}")
print(f"\nobs columns: {adata.obs.columns.tolist()}")
adata.write(OUTPUT_H5AD, compression='gzip')
print(f"Guardado en {OUTPUT_H5AD}")


Cargando modelo MOFA...


Factores: (1297, 30)
Índice ejemplo: ['4EGI-1_0.05_1', '9-ING-41_0.05_1', 'APTO-253_0.05_1']
Añadiendo MOA...
  MOA añadido. NaN: 0
Extrayendo pesos MOFA...
Tipo de W: <class 'pandas.core.frame.DataFrame'>
Creando AnnData...

AnnData object with n_obs × n_vars = 1297 × 30
    obs: 'drug', 'concentration', 'plate', 'moa-fine'
    uns: 'mofa_weights', 'mofa_weights_genes', 'mofa_weights_factors', 'mofa_views'

obs columns: ['drug', 'concentration', 'plate', 'moa-fine']
Guardado en /mnt/lustre/scratch/nlsas/home/ulc/co/mao/modelo_prueba_final_fast_convergence/adata/mofa_adata_30f.h5ad


In [ ]:
import duckdb as db
INPUT = "/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/datos/datos_con_placa_14/tidy_final.parquet"
OUTPUT = "'/mnt/netapp2/Store_uni/home/ulc/co/mao/TFM_final/datos/datos_con_placa_14/tidy_final_sin_group.parquet'"
db.query(f"""
COPY (
    SELECT feature, value, view, sample
    FROM parquet_scan('{INPUT}')
)
TO '{OUTPUT}' (FORMAT 'parquet', COMPRESSION 'gzip')
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

: 